## Synthetic Visit Occurrence — Liver Disease

### Ontology Lookup Service (OLS) via MCP

This notebook leverages the **OLS MCP service** (Ontology Lookup Service, exposed as Model Context Protocol tools) to source clinically accurate ICD-10 codes for liver disease. Starting from **DOID:409** (*liver disease*) in the Disease Ontology, the `getDescendants` and `searchClasses` MCP tools were used to traverse the ontology hierarchy and identify the ICD-10 codes in the **K70–K77** range — covering alcoholic liver disease, cirrhosis, steatotic liver disease, hepatic failure, and liver cancer. This ensures the synthetic data is grounded in a governed, standards-based classification rather than manually curated code lists.

### Certified Table Reference for Schema & Metadata

Genie Code resolved the `@visit_occurrence` table reference by following a **certified-first resolution** strategy:

1. **Search** — `tableSearch` identified candidate tables matching `visit_occurrence` across Unity Catalog.
2. **Certification check** — `readTable` with `extraDetails=true` confirmed that **`main.omop.visit_occurrence`** carries a **certified** governance tag, making it the authoritative source.
3. **Schema extraction** — The 17-column schema (types, nullability, column comments) was read from the certified table and faithfully mapped to a PySpark `StructType`.
4. **Metadata propagation** — After writing the synthetic DataFrame to `dmoore.project-2026-april.visit_occurrence`, all metadata was propagated from the certified source in separate DDL statements: table comment, column comments, table-level tags (`industry = HLS`), and column-level tags (`phi = true`).

This approach guarantees that derived tables inherit the governance posture of their certified source — comments, tags, and policies travel with the data.

### Workflow Diagram

![Practical Ontologies Workflow](https://raw.githubusercontent.com/dmoore247/practical-ontologies/refs/heads/main/images/databricks-ontology-architecture-diagram.png)

In [0]:
# Create a synthetic dataframe following the @visit_occurrence table schema,
# fill it with example data leveraging the ICD10 Ontologies for Liver Disease.
# ICD-10 codes sourced from OLS Disease Ontology descendants of DOID:409 (liver disease)
from pyspark.sql.functions import col, lit
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DateType,
    TimestampType, StringType
)
from datetime import datetime, date, timedelta
import random

# ----------------------------------------------------------------
# ICD-10 Liver Disease codes (K70-K77) mapped from Disease Ontology
# DOID:409  → liver disease
# DOID:0070658 → alcohol-associated liver disease
# DOID:5082 → liver cirrhosis
# DOID:9452 → steatotic liver disease
# DOID:3571 → liver cancer
# ----------------------------------------------------------------
liver_disease_icd10 = [
    ("K70.0",  "Alcoholic fatty liver"),
    ("K70.1",  "Alcoholic hepatitis"),
    ("K70.3",  "Alcoholic cirrhosis of liver"),
    ("K70.9",  "Alcoholic liver disease, unspecified"),
    ("K71.0",  "Toxic liver disease with cholestasis"),
    ("K72.0",  "Acute and subacute hepatic failure"),
    ("K72.1",  "Chronic hepatic failure"),
    ("K73.9",  "Chronic hepatitis, unspecified"),
    ("K74.0",  "Hepatic fibrosis"),
    ("K74.6",  "Other and unspecified cirrhosis of liver"),
    ("K75.4",  "Autoimmune hepatitis"),
    ("K76.0",  "Steatotic liver disease (NAFLD)"),
    ("K76.6",  "Portal hypertension"),
    ("K76.9",  "Liver disease, unspecified"),
    ("K77",    "Liver disorders in diseases classified elsewhere"),
]

# OMOP standard concept IDs
VISIT_CONCEPTS = {"IP": 9201, "OP": 9202, "ER": 9203}   # Inpatient / Outpatient / ER
VISIT_TYPE_CONCEPT_ID = 32817                              # EHR record

# Schema matching main.omop.visit_occurrence (17 columns)
schema = StructType([
    StructField("visit_occurrence_id",        IntegerType(),   False),
    StructField("person_id",                  IntegerType(),   False),
    StructField("visit_concept_id",           IntegerType(),   False),
    StructField("visit_start_date",           DateType(),      False),
    StructField("visit_start_datetime",       TimestampType(), True),
    StructField("visit_end_date",             DateType(),      False),
    StructField("visit_end_datetime",         TimestampType(), True),
    StructField("visit_type_concept_id",      IntegerType(),   False),
    StructField("provider_id",                IntegerType(),   True),
    StructField("care_site_id",               IntegerType(),   True),
    StructField("visit_source_value",         StringType(),    True),
    StructField("visit_source_concept_id",    IntegerType(),   True),
    StructField("admitted_from_concept_id",   IntegerType(),   True),
    StructField("admitted_from_source_value", StringType(),    True),
    StructField("discharged_to_concept_id",   IntegerType(),   True),
    StructField("discharged_to_source_value", StringType(),    True),
    StructField("preceding_visit_occurrence_id", IntegerType(), True),
])

# --- Generate 30 synthetic visits for liver-disease patients ----------
random.seed(42)
rows = []
base_date = date(2025, 1, 1)
vtype_keys = list(VISIT_CONCEPTS.keys())

for i in range(1, 31):
    person_id      = random.randint(1000, 1050)
    vtype_key      = random.choice(vtype_keys)
    visit_cid      = VISIT_CONCEPTS[vtype_key]

    start_offset   = random.randint(0, 364)
    start_date     = base_date + timedelta(days=start_offset)
    duration       = random.randint(1, 14) if vtype_key == "IP" else random.randint(0, 1)
    end_date       = start_date + timedelta(days=duration)

    start_dt       = datetime.combine(start_date, datetime.min.time().replace(
                        hour=random.randint(6, 22), minute=random.choice([0, 15, 30, 45])))
    end_dt         = datetime.combine(end_date, datetime.min.time().replace(
                        hour=random.randint(6, 22), minute=random.choice([0, 15, 30, 45])))

    icd_code, icd_label = random.choice(liver_disease_icd10)
    source_value   = f"{icd_code} {icd_label}"

    provider_id    = random.randint(100, 120) if random.random() > 0.1 else None
    care_site_id   = random.randint(1, 5)     if random.random() > 0.1 else None

    # Admitted-from / discharged-to only for inpatient visits
    adm_map  = {8536: "Home", 8844: "Emergency Room", 0: "Unknown"}
    disc_map = {8536: "Home", 8615: "Rehabilitation Facility", 0: "Unknown"}
    adm_cid  = random.choice(list(adm_map))  if vtype_key == "IP" else None
    disc_cid = random.choice(list(disc_map)) if vtype_key == "IP" else None

    preceding_id = i - 1 if i > 1 and random.random() > 0.7 else None

    rows.append((
        i, person_id, visit_cid,
        start_date, start_dt, end_date, end_dt,
        VISIT_TYPE_CONCEPT_ID,
        provider_id, care_site_id,
        source_value, 0,
        adm_cid, adm_map.get(adm_cid),
        disc_cid, disc_map.get(disc_cid),
        preceding_id,
    ))

visit_occurrence_df = spark.createDataFrame(rows, schema=schema)
display(visit_occurrence_df)

visit_occurrence_id,person_id,visit_concept_id,visit_start_date,visit_start_datetime,visit_end_date,visit_end_datetime,visit_type_concept_id,provider_id,care_site_id,visit_source_value,visit_source_concept_id,admitted_from_concept_id,admitted_from_source_value,discharged_to_concept_id,discharged_to_source_value,preceding_visit_occurrence_id
1,1040,9201,2025-01-13,2025-01-13T14:15:00.000Z,2025-01-25,2025-01-25T13:15:00.000Z,32817,117,null,K76.0 Steatotic liver disease (NAFLD),0,8844,Emergency Room,8536,Home,null
2,1001,9201,2025-04-22,2025-04-22T22:00:00.000Z,2025-04-26,2025-04-26T12:45:00.000Z,32817,108,1,"K70.9 Alcoholic liver disease, unspecified",0,8536,Home,0,Unknown,null
3,1017,9201,2025-04-21,2025-04-21T16:00:00.000Z,2025-05-04,2025-05-04T08:45:00.000Z,32817,111,1,K70.1 Alcoholic hepatitis,0,0,Unknown,8615,Rehabilitation Facility,null
4,1024,9201,2025-10-10,2025-10-10T17:15:00.000Z,2025-10-15,2025-10-15T08:00:00.000Z,32817,109,2,K75.4 Autoimmune hepatitis,0,8536,Home,8615,Rehabilitation Facility,null
5,1040,9202,2025-03-25,2025-03-25T17:15:00.000Z,2025-03-26,2025-03-26T14:00:00.000Z,32817,117,2,K74.6 Other and unspecified cirrhosis of liver,0,null,null,null,null,null
6,1017,9203,2025-12-19,2025-12-19T16:00:00.000Z,2025-12-19,2025-12-19T13:00:00.000Z,32817,108,null,K76.6 Portal hypertension,0,null,null,null,null,5
7,1036,9203,2025-06-11,2025-06-11T21:45:00.000Z,2025-06-11,2025-06-11T20:15:00.000Z,32817,117,5,K71.0 Toxic liver disease with cholestasis,0,null,null,null,null,null
8,1037,9202,2025-07-05,2025-07-05T10:45:00.000Z,2025-07-05,2025-07-05T08:00:00.000Z,32817,120,4,"K76.9 Liver disease, unspecified",0,null,null,null,null,null
9,1024,9202,2025-11-02,2025-11-02T22:30:00.000Z,2025-11-03,2025-11-03T06:00:00.000Z,32817,108,3,K75.4 Autoimmune hepatitis,0,null,null,null,null,null
10,1027,9201,2025-08-21,2025-08-21T14:15:00.000Z,2025-08-22,2025-08-22T22:00:00.000Z,32817,120,2,"K76.9 Liver disease, unspecified",0,8536,Home,8615,Rehabilitation Facility,9


In [0]:
# ----------------------------------------------------------------
# Save visit_occurrence_df to dmoore.`project-2026-april` schema
# with full metadata propagation from main.omop.visit_occurrence
# (table comment, column comments, table tags, column tags)
# ----------------------------------------------------------------

TARGET_TABLE = "`dmoore`.`project-2026-april`.`visit_occurrence`"

# 1. Write the DataFrame as a managed Delta table
visit_occurrence_df.write.mode("overwrite").saveAsTable(TARGET_TABLE)
print("Table created.")

# 2. Table comment (propagated from main.omop.visit_occurrence)
table_comment = (
    "This table records information about individual healthcare visits or "
    "encounters for patients. It contains data such as the start and end times "
    "of visits, the type of visit, the healthcare provider and facility involved, "
    "and details on admission and discharge sources and destinations. The table "
    "can be used to analyze patterns in patient visits, track sequences of care "
    "events, and understand the context of healthcare delivery across different "
    "providers and sites.\n\nAlso known as Patient Encounter."
)
spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS '{table_comment}'")
print("Table comment applied.")

# 3. Column comments (propagated from main.omop.visit_occurrence)
column_comments = {
    "visit_occurrence_id":        "Unique identifier for each individual healthcare visit or patient encounter.",
    "person_id":                  "Identifier linking the visit to the specific patient who experienced the healthcare encounter.",
    "visit_concept_id":           "Standardized concept identifier representing the type or nature of the visit.",
    "visit_start_date":           "Date on which the healthcare visit began, useful for daily level analysis.",
    "visit_start_datetime":       "Exact date and time marking the beginning of the healthcare encounter.",
    "visit_end_date":             "Date on which the healthcare visit concluded, useful for daily summaries.",
    "visit_end_datetime":         "Exact date and time when the healthcare visit ended.",
    "visit_type_concept_id":      "Concept identifier classifying the type of visit record, such as inpatient, outpatient, or emergency.",
    "provider_id":                "Identifier for the healthcare provider primarily responsible during the visit.",
    "care_site_id":               "Identifier for the healthcare facility or location where the patient was seen during the visit.",
    "visit_source_value":         "Original source code or description of the visit type as recorded in the source system.",
    "visit_source_concept_id":    "Source system concept identifier linked to the visit type for standardized mapping.",
    "admitted_from_concept_id":   "Concept identifier indicating the source or location from which the patient was admitted for this visit.",
    "admitted_from_source_value": "Source system description of the admission origin, providing contextual admission details.",
    "discharged_to_concept_id":   "Concept identifier representing the destination to which the patient was discharged after the visit.",
    "discharged_to_source_value": "Source system description of the discharge destination, detailing post-visit patient disposition.",
    "preceding_visit_occurrence_id": "Identifier of the immediate prior visit to establish visit sequences or continuity of care.",
}
for col_name, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN `{col_name}` COMMENT '{comment}'")
print(f"Column comments applied ({len(column_comments)} columns).")

# 4. Table-level tags (propagated from main.omop.visit_occurrence)
spark.sql(f"ALTER TABLE {TARGET_TABLE} SET TAGS ('industry' = 'HLS')")
print("Table tags applied.")

# 5. Column-level tags — phi (propagated from main.omop.visit_occurrence)
for col_name in column_comments:
    spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN `{col_name}` SET TAGS ('phi' = 'true')")
print(f"Column phi tags applied ({len(column_comments)} columns).")

print(f"\nDone — {TARGET_TABLE} is ready with full metadata.")

Table created.
Table comment applied.
Column comments applied (17 columns).
Table tags applied.
Column phi tags applied (17 columns).

Done — `dmoore`.`project-2026-april`.`visit_occurrence` is ready with full metadata.


In [0]:
%sql describe extended dmoore.`project-2026-april`.visit_occurrence

col_name,data_type,comment
visit_occurrence_id,int,Unique identifier for each individual healthcare visit or patient encounter.
person_id,int,Identifier linking the visit to the specific patient who experienced the healthcare encounter.
visit_concept_id,int,Standardized concept identifier representing the type or nature of the visit.
visit_start_date,date,"Date on which the healthcare visit began, useful for daily level analysis."
visit_start_datetime,timestamp,Exact date and time marking the beginning of the healthcare encounter.
visit_end_date,date,"Date on which the healthcare visit concluded, useful for daily summaries."
visit_end_datetime,timestamp,Exact date and time when the healthcare visit ended.
visit_type_concept_id,int,"Concept identifier classifying the type of visit record, such as inpatient, outpatient, or emergency."
provider_id,int,Identifier for the healthcare provider primarily responsible during the visit.
care_site_id,int,Identifier for the healthcare facility or location where the patient was seen during the visit.


In [0]:
%sql
SELECT 'table' AS level, NULL AS column_name, tag_name, tag_value
FROM system.information_schema.table_tags
WHERE catalog_name = 'dmoore'
  AND schema_name = 'project-2026-april'
  AND table_name = 'visit_occurrence'
UNION ALL
SELECT 'column' AS level, column_name, tag_name, tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'dmoore'
  AND schema_name = 'project-2026-april'
  AND table_name = 'visit_occurrence'
ORDER BY level, column_name

level,column_name,tag_name,tag_value
column,admitted_from_concept_id,phi,true
column,admitted_from_source_value,phi,true
column,care_site_id,phi,true
column,discharged_to_concept_id,phi,true
column,discharged_to_source_value,phi,true
column,person_id,phi,true
column,preceding_visit_occurrence_id,phi,true
column,provider_id,phi,true
column,visit_concept_id,phi,true
column,visit_end_date,phi,true
